 # Analyse qualitative des sources bibliographiques — POC Data

 Objectif :

 - consolider les résultats déjà exportés ;
 - comparer Google Books, OpenLibrary, BNF et Nudger ;
 - analyser la couverture ISBN ;
 - analyser la complétude des métadonnées ;
 - identifier les forces et limites de chaque source ;
 - préparer le futur benchmark étendu.

In [1]:

from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

## Chargement des exports

In [2]:

paths = {
    "google_books": PROJECT_ROOT / "data" / "processed" / "google_books" / "google_books_results_full.csv",
    "openlibrary": PROJECT_ROOT / "data" / "processed" / "openlibrary" / "openlibrary_results_full.csv",
    "bnf": PROJECT_ROOT / "data" / "processed" / "bnf" / "bnf_results_full.csv",
    "nudger": PROJECT_ROOT / "data" / "processed" / "nudger" / "nudger_results_full.csv",
}

dfs = []

for source, path in paths.items():
    df = pd.read_csv(path)
    df["source"] = source
    dfs.append(df)

df_results = pd.concat(dfs, ignore_index=True)

df_results.head()

,source,isbn_query,found,error,source_id,google_books_id,isbn,title,subtitle,authors,...,openlibrary_key,publishers,format,types,rights,url,offers_count,min_price,currency,last_updated
0,google_books,9782351423554,True,NaN,f4RwPgAACAAJ,f4RwPgAACAAJ,9.782351e+12,Vinland Saga,NaN,"['Makoto Yukimura', 'Xavière Daumarie']",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,google_books,9791042019396,False,no_result,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,google_books,9782820354488,False,no_result,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,google_books,9782290042298,True,NaN,U4qHPQAACAAJ,U4qHPQAACAAJ,9.782290e+12,Le précepteur du héros,NaN,['Riku Sanjô'],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,google_books,9782845808331,True,NaN,xaBwPAAACAAJ,xaBwPAAACAAJ,9.782846e+12,Dragon quest : la quête de Daï,NaN,"['Riku Sanjô', 'Kōji Inada, 1964-']",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Couverture par source

In [3]:
# %%
coverage_by_source = (
    df_results
    .groupby("source")["found"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="coverage_rate")
    .sort_values("coverage_rate", ascending=False)
)

coverage_by_source

,source,coverage_rate
2,nudger,95.0
1,google_books,50.0
0,bnf,45.0
3,openlibrary,20.0


Les premiers benchmarks mettent en évidence des **écarts importants** entre les sources évaluées.

*Nudger* obtient la **meilleure couverture** avec 95 % des ISBN retrouvés. *Google Book*s (50 %) et *BNF* (45 %) affichent des résultats relativement proches et semblent davantage adaptés à l’enrichissement des métadonnées. OpenLibrary présente une couverture plus limitée sur l’échantillon testé (20 %).

Ces résultats constituent une première tendance mais devront être confirmés sur un échantillon plus large avant de définir la stratégie multi-sources de CollectionLens.

## Complétude des métadonnées par source

In [4]:
metadata_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "language",
    "description",
    "categories",
    "thumbnail",
]

available_fields = [
    field for field in metadata_fields
    if field in df_results.columns
]

metadata_completeness = (
    df_results
    .groupby("source")[available_fields]
    .apply(lambda df: df.notna().mean())
    .mul(100)
    .round(2)
)

metadata_completeness

,title,authors,publisher,published_date,language,description,categories,thumbnail
source,,,,,,,,
bnf,45.0,45.0,45.0,45.0,45.0,45.0,45.0,0.0
google_books,50.0,50.0,10.0,50.0,50.0,20.0,50.0,5.0
nudger,95.0,95.0,55.0,0.0,0.0,0.0,95.0,0.0
openlibrary,20.0,20.0,20.0,20.0,0.0,0.0,20.0,15.0


L’analyse de la complétude des métadonnées confirme que chaque source possède des forces différentes.

Nudger se distingue par une **excellente couverture** des champs essentiels à l’identification des ouvrages (titre, auteurs, catégories), mais ne fournit **quasiment aucune donnée d’enrichissement** comme les descriptions, langues ou dates de publication.

BNF présente une **complétude homogène** sur l’ensemble des métadonnées disponibles et constitue la source la plus équilibrée parmi les APIs testées. La présence de descriptions représente un atout intéressant pour les futurs usages de recommandation et de RAG.

Google Books fournit également plusieurs métadonnées utiles mais avec une qualité plus variable selon les champs, n**otamment pour les éditeurs et les descriptions**.

OpenLibrary apparaît comme la source la **moins complète** sur l’échantillon étudié.

Ces résultats renforcent l’hypothèse qu’une stratégie multi-sources sera nécessaire afin de combiner la forte couverture de Nudger avec les capacités d’enrichissement offertes par BNF et Google Books.

## Couverture cumulée multi-sources

In [5]:

df_coverage_by_isbn = (
    df_results
    .pivot_table(
        index="isbn_query",
        columns="source",
        values="found",
        aggfunc="first",
    )
    .reset_index()
)

source_columns = [
    col for col in df_coverage_by_isbn.columns
    if col != "isbn_query"
]

df_coverage_by_isbn[source_columns] = (
    df_coverage_by_isbn[source_columns]
    .fillna(False)
    .astype(bool)
)

df_coverage_by_isbn["sources_found_count"] = (
    df_coverage_by_isbn[source_columns].sum(axis=1)
)

df_coverage_by_isbn["found_anywhere"] = (
    df_coverage_by_isbn["sources_found_count"] > 0
)

global_coverage = round(
    df_coverage_by_isbn["found_anywhere"].mean() * 100,
    2,
)

print(f"Couverture cumulée multi-sources : {global_coverage:.2f}%")

df_coverage_by_isbn

Couverture cumulée multi-sources : 100.00%


source,isbn_query,bnf,google_books,nudger,openlibrary,sources_found_count,found_anywhere
0,9782203001190,False,True,True,True,3,True
1,9782290042298,False,True,True,True,3,True
2,9782351423554,True,True,True,True,4,True
3,9782374123035,False,False,True,False,1,True
4,9782374126173,False,False,False,True,1,True
5,9782384967421,False,False,True,False,1,True
6,9782413042341,True,True,True,False,3,True
7,9782820354488,True,False,True,False,2,True
8,9782822244787,True,True,True,False,3,True
9,9782845808331,True,True,True,False,3,True


La combinaison des différentes sources permet d’atteindre une **couverture cumulée de 100 %** sur l’échantillon testé, ce qui confirme l'intérêt d'une approche multi-sources pour maximiser la récupération des métadonnées.

## Sources les plus contributives

In [6]:
df_coverage_by_isbn["best_source_hint"] = df_coverage_by_isbn[source_columns].idxmax(axis=1)

df_coverage_by_isbn[
    ["isbn_query", "sources_found_count", "found_anywhere", "best_source_hint"]
]

source,isbn_query,sources_found_count,found_anywhere,best_source_hint
0,9782203001190,3,True,google_books
1,9782290042298,3,True,google_books
2,9782351423554,4,True,bnf
3,9782374123035,1,True,nudger
4,9782374126173,1,True,openlibrary
5,9782384967421,1,True,nudger
6,9782413042341,3,True,bnf
7,9782820354488,2,True,bnf
8,9782822244787,3,True,bnf
9,9782845808331,3,True,bnf


## Analyse des erreurs

In [7]:

error_summary = (
    df_results[df_results["found"] == False]
    .groupby(["source", "error"])
    .size()
    .reset_index(name="count")
    .sort_values(["source", "count"], ascending=[True, False])
)

error_summary

,source,error,count
0,bnf,no_result,11
1,google_books,no_result,10
2,nudger,no_result,1
3,openlibrary,no_result,16


## Exemples qualitatifs par source

### google books

In [8]:
qualitative_fields = [
    "isbn_query",
    "title",
    "authors",
    "publisher",
    "published_date",
    "description",
    "categories",
]

In [9]:
df_results[
    (df_results["source"] == "google_books")
    & (df_results["found"] == True)
][qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,categories
0,9782351423554,Vinland Saga,"['Makoto Yukimura', 'Xavière Daumarie']",NaN,2009-01-15,"Depuis qu'Askeladd, un chef de guerre fourbe e...",[]
3,9782290042298,Le précepteur du héros,['Riku Sanjô'],NaN,1999-01-01,NaN,[]
4,9782845808331,Dragon quest : la quête de Daï,"['Riku Sanjô', 'Kōji Inada, 1964-']",NaN,2007-01-10,Depuis la fin de la guerre contre le Roi du Ma...,[]
5,9782413042341,Dragon Quest Tome 1,[],NaN,2022-03-02,NaN,[]
10,9791039143431,La citadelle hurlante,[],NaN,2026-03-25,NaN,[]
11,9791026854920,Harley Quinn,[],NaN,2026-02-13,NaN,[]
15,9782203001190,Tintin au Tibet,['Hergé'],Editions Moulinsart,1960-01-01,Un avion de ligne à bord duquel le jeune Chino...,['Young Adult Nonfiction']
16,9782858150045,Titine au bistrot,['Yan Lindingre'],Fluide glacial - Audie,2007,"Si, quand on vous dit Titine vous pensez Marti...",[]
17,9791038206229,Fleurs de pavés,['Sylvain Frécon'],NaN,2024-02-07,NaN,[]
18,9782822244787,Stranger Trucs,[],NaN,2024-11-07,NaN,[]


**L’analyse qualitative de Google Books** montre que la source fournit généralement des titres fiables ainsi que des dates de publication bien renseignées. Certaines notices contiennent également des descriptions détaillées, ce qui constitue un atout intéressant pour les futurs usages de recommandation et de RAG.

En revanche, plusieurs limites apparaissent sur cet échantillon :
- de nombreux éditeurs sont absents ;
- les catégories sont rarement renseignées ou peu pertinentes ;
- certains auteurs sont manquants ;
- la qualité des métadonnées varie fortement d'une notice à l'autre.

Google Books semble donc davantage adapté à **l’enrichissement de certaines métadonnées** (titres, dates, descriptions) qu’à une couverture bibliographique complète utilisée seule.

### BNF

In [10]:
df_results[
    (df_results["source"] == "bnf")
    & (df_results["found"] == True)
][qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,categories
40,9782351423554,Vinland saga. 1 / Makoto Yukimura,"['Yukimura, Makoto (1976-....). Auteur du texte']",Kurokawa (Paris),2009.0,Collection : Collection dirigée par Grégoire H...,[]
42,9782820354488,Blue Exorcist. 32,"['Katō, Kazue (1980-....)']",Crunchyroll (Paris),2026.0,Collection : Shônen up !,[]
44,9782845808331,"Dragon quest : la quête de Daï. 1 / scénario, ...","['Sanjō, Riku (1964-....). Auteur du texte', '...",Éd. Tonkam (Paris),2007.0,Collection : Shonen,[]
45,9782413042341,"Dragon quest, the adventure of Dai. 1, Les dis...","['Sanjō, Riku (1964-....). Auteur du texte']",Delcourt-Tonkam (Paris),2022.0,Code à barres commercial : EAN 9782413042341,[]
50,9791039143431,Star Wars : La citadelle hurlante,"['Aaron, Jason (1973-....)', 'Gillen, Kieron (...",Panini France (Nice),2026.0,Collection : Marvel poche,[]
52,9791039140652,La Chose ([Éd. spéciale] exclusivité FNAC) [sc...,"['Johns, Geoff (1973-....). Auteur du texte', ...",Panini comics (Nice),2025.0,Code à barres commercial : EAN 9791039140652,[]
56,9782858150045,Titine au bistrot / Lindingre,"['Lindingre, Yan (1969-....). Auteur du texte']","Audie-""Fluide glacial"" (Paris)",2007.0,Code à barres commercial : EAN 9782858150045,[]
57,9791038206229,"Fleurs de pavés / scénario, dessin, couleurs, ...","['Frécon, Sylvain (1972-....). Auteur du texte']","""Fluide glacial"" (Paris)",2024.0,Code à barres commercial : EAN 9791038206229,[]
58,9782822244787,"Stranger trucs / Antoine Piers, Novy ; [dessin...","['Piers, Antoine. Auteur du texte', 'Novy (196...",Jungle (Paris),2024.0,Code à barres commercial : EAN 9782822244787,[]


L’analyse qualitative de la BNF met en évidence des métadonnées particulièrement riches et cohérentes pour les ouvrages français.

Les titres, auteurs, éditeurs et dates de publication sont généralement bien renseignés et souvent plus précis que dans les autres sources. Les notices contiennent également des informations bibliographiques complémentaires, comme les collections, les éditions ou les contributeurs détaillés.

Les descriptions sont présentes sur la majorité des ouvrages retrouvés, même si elles correspondent souvent à des informations de catalogage (collection, EAN, mentions d’édition) plutôt qu’à de véritables résumés éditoriaux.

En revanche, les catégories sont absentes sur l’ensemble de l’échantillon étudié et les données nécessiteront probablement un travail de normalisation afin de simplifier les intitulés d’auteurs, de titres et de descriptions.

La BNF apparaît ainsi comme une excellente source de métadonnées bibliographiques fiables pour les ouvrages français, particulièrement adaptée à l’enrichissement et à la validation des données de CollectionLens.

### Open library

In [11]:
df_results[
    (df_results["source"] == "openlibrary")
    & (df_results["found"] == True)
][qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,categories
20,9782351423554,Vinland Saga (1),['幸村誠 (Makoto Yukimura)'],KUROKAWA,2009-01-15,NaN,"['franchise:ヴィンランド・サガ', 'series:ヴィンランド・サガ', 'f..."
23,9782290042298,"Fly, tome 1",[],J'ai lu,"January 4, 1999",NaN,[]
29,9782374126173,"Sinbad, Tome 1",['Atsuji Yamamoto'],Black Box,2025,NaN,['Manga']
35,9782203001190,Tintin au Tibet,['Hergé'],Casterman,1960,NaN,"['series:Les Aventures de Tintin', 'Fiction', ..."


L’analyse qualitative d’OpenLibrary montre une couverture limitée sur l’échantillon étudié, mais les notices retrouvées sont globalement cohérentes.

Les titres, éditeurs et dates de publication sont généralement corrects lorsque l’ouvrage est présent dans la base. Certaines notices proposent également des catégories intéressantes, notamment pour les mangas ou les séries connues.

En revanche, plusieurs limites apparaissent :
- très peu d’ouvrages retrouvés ;
- descriptions absentes sur l’ensemble des notices analysées ;
- auteurs parfois manquants ;
- qualité et structure des catégories hétérogènes ;
- métadonnées variables selon les ouvrages.

OpenLibrary présente une faible couverture sur le périmètre étudié. En revanche, les notices retrouvées offrent une richesse sémantique importante qui pourrait s'avérer particulièrement utile pour les futurs systèmes de recommandation et de RAG.

La source apparaît davantage comme une source d'enrichissement que comme une source principale de récupération des métadonnées.

### Nugder

L’analyse qualitative de Nudger confirme son excellente couverture des ISBN sur l’échantillon étudié. La quasi-totalité des ouvrages testés ont été retrouvés, quel que soit leur type (manga, BD ou comics).

Les titres sont généralement présents et suffisamment précis pour identifier correctement les ouvrages. Les éditeurs sont également souvent renseignés lorsqu'ils sont disponibles dans la source.

En revanche, les métadonnées d’enrichissement sont quasiment absentes :
- auteurs ;
- dates de publication ;
- descriptions ;
- catégories exploitables.

Les données semblent principalement orientées catalogue produit et identification d’ouvrage plutôt qu’enrichissement bibliographique.

Nudger apparaît ainsi comme une excellente source de couverture et de résolution ISBN, mais devra être complétée par d’autres sources pour obtenir des métadonnées plus riches et adaptées aux futurs usages de recommandation et de RAG.

In [12]:
df_results[
    (df_results["source"] == "nudger")
    & (df_results["found"] == True)
][qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,categories
60,9782351423554,vinland saga - tome,[],[Kurokawa],NaN,NaN,[]
61,9791042019396,Vinland Saga - Tome 29,[],NaN,NaN,NaN,[]
62,9782820354488,Blue Exorcist T32,[],NaN,NaN,NaN,[]
63,9782290042298,fly - tome 1 : le précepteur du héros,[],[J'ai lu],NaN,NaN,[]
64,9782845808331,Dragon Quest Tome 1,[],[Tonkam],NaN,NaN,[]
65,9782413042341,dragon quest - the adventure of daï t01,[],[Delcourt / Tonkam],NaN,NaN,[]
66,9782384967421,Les quatre frères Yuzuki Tome 15 (Manga),[],NaN,NaN,NaN,[]
67,9791043301087,Air Gear Unlimited Tome 07 (Manga),[],NaN,NaN,NaN,[]
68,9782374123035,bioman,[],[Black Box Editions],NaN,NaN,[]
70,9791039143431,Star Wars : La citadelle hurlante,[],[Panini comics],NaN,NaN,[]


##Conclusion intermédiaire

Les premiers benchmarks réalisés sur un échantillon de 20 ISBN ont permis d’identifier des différences significatives entre les sources évaluées.

La couverture cumulée des sources atteint 100 %, ce qui confirme l’intérêt d’une approche multi-sources pour maximiser la récupération des métadonnées bibliographiques.

Les résultats mettent en évidence des rôles complémentaires :

- **Nudger** se distingue par une couverture exceptionnelle (95 %) et constitue actuellement le meilleur candidat pour l’identification et la résolution des ISBN.
- **BNF** fournit les métadonnées bibliographiques les plus structurées et les plus fiables pour les ouvrages français, avec une bonne qualité sur les auteurs, éditeurs, dates et descriptions.
- **Google Books** apporte un enrichissement intéressant grâce à certaines descriptions détaillées et à des métadonnées complémentaires, mais présente une qualité plus variable selon les ouvrages.
- **OpenLibrary** affiche une couverture plus faible sur le périmètre manga, BD et comics étudié, mais peut rester une source de secours ponctuelle.

Cette première analyse suggère qu’aucune source ne répond seule à l’ensemble des besoins de CollectionLens. Une stratégie multi-sources semble donc nécessaire afin de combiner :

- la couverture de Nudger ;
- la qualité bibliographique de la BNF ;
- les enrichissements de Google Books ;
- un éventuel fallback via OpenLibrary.

Ces conclusions restent toutefois préliminaires. Elles devront être validées sur un échantillon plus large et plus représentatif de la collection réelle avant de définir la stratégie d’agrégation définitive et les règles de fusion des métadonnées.

## Audit des champs disponibles dans les données brutes

In [13]:
df_results.columns.tolist()

['source',
 'isbn_query',
 'found',
 'error',
 'source_id',
 'google_books_id',
 'isbn',
 'title',
 'subtitle',
 'authors',
 'publisher',
 'published_date',
 'language',
 'description',
 'categories',
 'thumbnail',
 'page_count',
 'print_type',
 'maturity_rating',
 'average_rating',
 'ratings_count',
 'preview_link',
 'info_link',
 'canonical_volume_link',
 'industry_identifiers',
 'raw_data',
 'status_code',
 'openlibrary_key',
 'publishers',
 'format',
 'types',
 'rights',
 'url',
 'offers_count',
 'min_price',
 'currency',
 'last_updated']

### Voir les champs non vides par source

In [14]:
for source in df_results["source"].unique():
    df_source = df_results[
        (df_results["source"] == source)
        & (df_results["found"] == True)
    ]

    non_empty_rates = (
        df_source
        .notna()
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
    )

    print("=" * 80)
    print(source.upper())
    print("=" * 80)

    display(non_empty_rates[non_empty_rates > 0])

GOOGLE_BOOKS


source                   100.0
isbn_query               100.0
found                    100.0
source_id                100.0
google_books_id          100.0
maturity_rating          100.0
isbn                     100.0
title                    100.0
authors                  100.0
language                 100.0
published_date           100.0
print_type               100.0
categories               100.0
industry_identifiers     100.0
canonical_volume_link    100.0
status_code              100.0
raw_data                 100.0
info_link                100.0
preview_link             100.0
page_count                90.0
description               40.0
publisher                 20.0
thumbnail                 10.0
dtype: float64

OPENLIBRARY


source                   100.0
isbn_query               100.0
found                    100.0
source_id                100.0
isbn                     100.0
publishers               100.0
title                    100.0
publisher                100.0
authors                  100.0
published_date           100.0
categories               100.0
print_type               100.0
preview_link             100.0
industry_identifiers     100.0
openlibrary_key          100.0
status_code              100.0
raw_data                 100.0
info_link                100.0
canonical_volume_link    100.0
thumbnail                 75.0
page_count                50.0
subtitle                  25.0
dtype: float64

BNF


source                  100.0
isbn_query              100.0
found                   100.0
source_id               100.0
isbn                    100.0
print_type              100.0
title                   100.0
publisher               100.0
authors                 100.0
published_date          100.0
language                100.0
categories              100.0
description             100.0
format                  100.0
status_code             100.0
rights                  100.0
types                   100.0
raw_data                100.0
industry_identifiers    100.0
dtype: float64

NUDGER


source                   100.00
isbn_query               100.00
found                    100.00
source_id                100.00
isbn                     100.00
url                      100.00
title                    100.00
authors                  100.00
categories               100.00
industry_identifiers     100.00
canonical_volume_link    100.00
info_link                100.00
preview_link             100.00
last_updated             100.00
offers_count             100.00
raw_data                 100.00
min_price                 89.47
currency                  89.47
page_count                57.89
publisher                 57.89
format                    52.63
print_type                52.63
dtype: float64

In [15]:
import ast
from pprint import pprint


def parse_raw_data(value):
    """
    Convertit raw_data en objet Python quand il est stocké en string CSV.
    """
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return value

    return value

### Google books

L'analyse des données brutes de Google Books montre que la majorité des champs pertinents sont déjà récupérés dans la normalisation actuelle.

Les notices contiennent des informations riches telles que les titres, auteurs, descriptions, dates de publication, nombre de pages, catégories, identifiants ISBN et liens vers les fiches Google Books.

Quelques champs complémentaires méritent néanmoins une analyse plus approfondie, notamment les extraits (`textSnippet`), les miniatures (`imageLinks`) ainsi que d'éventuelles informations de notation ou de série présentes sur certaines notices.

À ce stade, Google Books apparaît comme une source déjà bien exploitée, avec un potentiel d'enrichissement supplémentaire relativement limité par rapport à d'autres sources comme la BNF.

In [16]:
raw_google = (
    df_results[
        (df_results["source"] == "google_books")
        & (df_results["found"] == True)
    ]["raw_data"]
    .iloc[0]
)

raw_google = parse_raw_data(raw_google)

pprint(raw_google)

{'accessInfo': {'accessViewStatus': 'NONE',
                'country': 'FR',
                'embeddable': False,
                'epub': {'isAvailable': False},
                'pdf': {'isAvailable': False},
                'publicDomain': False,
                'quoteSharingAllowed': False,
                'textToSpeechPermission': 'ALLOWED',
                'viewability': 'NO_PAGES',
                'webReaderLink': 'http://play.google.com/books/reader?id=f4RwPgAACAAJ&hl=&as_pt=BOOKS&source=gbs_api'},
 'etag': 'n6Og3US3oXU',
 'id': 'f4RwPgAACAAJ',
 'kind': 'books#volume',
 'saleInfo': {'country': 'FR', 'isEbook': False, 'saleability': 'NOT_FOR_SALE'},
 'searchInfo': {'textSnippet': 'Vol. 15: Thorfinn et Einar, enfin libres, '
                               'retournent avec Leif Eriksson en Islande, '
                               'terre natale de Thorfinn, dans l&#39;espoir '
                               'd&#39;y trouver le financement nécessaire à '
                             

### BNF

L'analyse des données brutes de la BNF montre que la normalisation actuelle exploite déjà la majorité des informations bibliographiques disponibles.

Quelques champs complémentaires pourraient être ajoutés, notamment les informations de format physique (nombre de pages, dimensions), les identifiants ARK de la BNF ainsi que certaines métadonnées de gestion des notices (dates de création et de mise à jour).

Les notices BNF apparaissent particulièrement adaptées à la validation et à l'enrichissement bibliographique des ouvrages français. En revanche, les données observées contiennent peu de contenus éditoriaux détaillés et semblent davantage orientées catalogage que recommandation ou RAG.

In [17]:
raw_bnf = (
    df_results[
        (df_results["source"] == "bnf")
        & (df_results["found"] == True)
    ]["raw_data"]
    .iloc[0]
)

raw_bnf = parse_raw_data(raw_bnf)

pprint(raw_bnf)

{'srw:extraRecordData': {'ixm:attr': [{'#text': '20081230',
                                       '@name': 'CreationDate'},
                                      {'#text': '20140705',
                                       '@name': 'LastModificationDate'}],
                         'mn:score': '6.4417996'},
 'srw:recordData': {'oai_dc:dc': {'@xmlns:dc': 'http://purl.org/dc/elements/1.1/',
                                  '@xmlns:oai_dc': 'http://www.openarchives.org/OAI/2.0/oai_dc/',
                                  '@xmlns:xsi': 'http://www.w3.org/2001/XMLSchema-instance',
                                  '@xsi:schemaLocation': 'http://www.openarchives.org/OAI/2.0/oai_dc/ '
                                                         'http://www.openarchives.org/OAI/2.0/oai_dc.xsd',
                                  'dc:creator': 'Yukimura, Makoto (1976-....). '
                                                'Auteur du texte',
                                  'dc:date': '2009',
    

### open library

In [18]:
raw_openlibrary = (
    df_results[
        (df_results["source"] == "openlibrary")
        & (df_results["found"] == True)
    ]["raw_data"]
    .iloc[0]
)

raw_openlibrary = parse_raw_data(raw_openlibrary)

pprint(raw_openlibrary)

{'authors': [{'name': '幸村誠 (Makoto Yukimura)',
              'url': 'http://openlibrary.org/authors/OL1439316A/幸村誠_(Makoto_Yukimura)'}],
 'cover': {'large': 'https://covers.openlibrary.org/b/id/14839285-L.jpg',
           'medium': 'https://covers.openlibrary.org/b/id/14839285-M.jpg',
           'small': 'https://covers.openlibrary.org/b/id/14839285-S.jpg'},
 'identifiers': {'isbn_10': ['2351423550'],
                 'isbn_13': ['9782351423554'],
                 'openlibrary': ['OL57621241M']},
 'key': '/books/OL57621241M',
 'publish_date': '2009-01-15',
 'publish_places': [{'name': 'France'}],
 'publishers': [{'name': 'KUROKAWA'}],
 'subject_people': [{'name': 'Lief Ericson',
                     'url': 'https://openlibrary.org/subjects/person:lief_ericson'},
                    {'name': 'Thorfinn',
                     'url': 'https://openlibrary.org/subjects/person:thorfinn'},
                    {'name': 'Thors',
                     'url': 'https://openlibrary.org/subjects/perso

L'analyse des données brutes d'OpenLibrary révèle une richesse sémantique supérieure à celle observée dans les autres sources testées.

Les notices peuvent contenir de nombreuses informations exploitables pour la recommandation et le futur système RAG, notamment des genres, thématiques, publics cibles, personnages, lieux et périodes historiques associés aux œuvres.

La présence de catégories détaillées telles que "seinen", "shōnen", "historical", "adventure" ou "graphic novel" représente un potentiel particulièrement intéressant pour les futurs algorithmes de recommandation.

OpenLibrary fournit également plusieurs tailles de couverture ainsi que des métadonnées structurées autour des sujets, personnages et univers des œuvres.

Malgré cette richesse, la faible couverture observée sur l'échantillon étudié limite actuellement son rôle à une source complémentaire d'enrichissement plutôt qu'à une source principale de métadonnées.

## Synthèse des champs disponibles

| Champ métier      | Google Books             | BNF                        | OpenLibrary    | Nudger                   | Intérêt métier |
| ----------------- | ------------------------ | -------------------------- | -------------- | ------------------------ | -------------- |
| title             | Disponible               | Disponible                 | Disponible     | Disponible               | Critique       |
| authors           | Disponible               | Disponible                 | Disponible     | Non disponible           | Critique       |
| publisher         | Partiellement disponible | Disponible                 | Disponible     | Partiellement disponible | Moyen          |
| published_date    | Disponible               | Disponible                 | Disponible     | Non disponible           | Moyen          |
| description       | Partiellement disponible | Partiellement disponible   | Non disponible | Non disponible           | Fort           |
| categories        | Partiellement disponible | Non disponible             | Disponible     | Partiellement disponible | Fort           |
| subjects          | Non disponible           | Non disponible             | Disponible     | Non disponible           | Fort           |
| subject_people    | Non disponible           | Non disponible             | Disponible     | Non disponible           | Fort           |
| subject_places    | Non disponible           | Non disponible             | Disponible     | Non disponible           | Moyen          |
| subject_times     | Non disponible           | Non disponible             | Disponible     | Non disponible           | Moyen          |
| thumbnail / cover | Partiellement disponible | Non disponible             | Disponible     | Non disponible           | Moyen          |
| page_count        | Disponible               | Potentiellement disponible | Non disponible | Non disponible           | Faible         |
| bnf_ark           | Non disponible           | Disponible                 | Non disponible | Non disponible           | Moyen          |
| textSnippet       | Disponible               | Non disponible             | Non disponible | Non disponible           | Moyen          |


## Décisions issues de l'audit

| Champ                      | Décision  | Justification              |
| -------------------------- | --------- | -------------------------- |
| title                      | Conserver | Champ essentiel            |
| authors                    | Conserver | Champ essentiel            |
| publisher                  | Conserver | Métadonnée importante      |
| published_date             | Conserver | Tri et filtrage            |
| description                | Conserver | Recommandation et RAG      |
| categories                 | Conserver | Recommandation             |
| subjects OpenLibrary       | Intégrer  | Forte valeur sémantique    |
| subject_people OpenLibrary | Intégrer  | Univers et personnages     |
| subject_places OpenLibrary | À étudier | Potentiel pour le RAG      |
| subject_times OpenLibrary  | À étudier | Potentiel historique       |
| thumbnail / cover          | Conserver | Interface utilisateur      |
| page_count                 | À étudier | Information complémentaire |
| bnf_ark                    | Intégrer  | Identifiant externe fiable |
| textSnippet Google Books   | À étudier | Fallback de description    |


## Décisions retenues

L’audit des champs disponibles montre que les sources ont des rôles complémentaires.

Google Books est déjà correctement exploité pour les métadonnées classiques : titre, auteurs, date, langue, nombre de pages, description et liens externes. Quelques champs complémentaires comme `textSnippet` pourront être étudiés comme fallback de description.

La BNF apporte des métadonnées bibliographiques fiables pour les ouvrages français. Elle pourra être enrichie avec des identifiants ARK et éventuellement des informations extraites du format physique.

OpenLibrary présente une faible couverture mais une richesse sémantique importante lorsque les notices sont disponibles. Les champs `subjects`, `subject_people`, `subject_places` et `subject_times` sont particulièrement intéressants pour les futurs usages de recommandation et de RAG.

Nudger reste la meilleure source de couverture ISBN, mais ses données sont davantage orientées identification produit que description ou enrichissement sémantique.

Avant d’étendre le benchmark à un échantillon plus large, il sera pertinent d’améliorer la normalisation des champs à forte valeur métier, notamment ceux provenant d’OpenLibrary et de la BNF.

## Prochaines étapes

Suite à cette analyse qualitative :

- enrichir les fonctions de normalisation lorsque cela présente un intérêt métier ;
- intégrer certains champs complémentaires identifiés lors de l'audit ;
- constituer un échantillon étendu issu de la collection réelle ;
- réaliser un benchmark multi-sources sur plusieurs centaines d'ouvrages ;
- définir la future stratégie d'agrégation des métadonnées.

## Améliorations de normalisation retenues

L’audit des données brutes permet d’identifier plusieurs améliorations prioritaires avant de passer au benchmark étendu.

Les améliorations retenues sont :

| Source | Amélioration | Intérêt |
|---|---|---|
| OpenLibrary | Ajouter `subjects` | Recommandation et RAG |
| OpenLibrary | Ajouter `subject_people` | Personnages et univers |
| OpenLibrary | Ajouter `subject_places` | Contexte géographique |
| OpenLibrary | Ajouter `subject_times` | Contexte temporel |
| OpenLibrary | Ajouter `cover_small`, `cover_medium`, `cover_large` | Interface utilisateur |
| BNF | Ajouter `bnf_ark` | Identifiant externe fiable |
| BNF | Étudier l’extraction du nombre de pages depuis `format` | Métadonnée complémentaire |
| Google Books | Étudier `textSnippet` comme fallback de description | Enrichissement textuel |

Ces améliorations seront intégrées avant le benchmark étendu afin de ne pas sous-évaluer la valeur réelle de chaque source.

In [28]:
normalization_improvements = pd.DataFrame(
    [
        {
            "source": "openlibrary",
            "field": "subjects",
            "status": "to_integrate",
            "interest": "recommendation / rag",
        },
        {
            "source": "openlibrary",
            "field": "subject_people",
            "status": "to_integrate",
            "interest": "characters / universe",
        },
        {
            "source": "openlibrary",
            "field": "subject_places",
            "status": "to_integrate",
            "interest": "context",
        },
        {
            "source": "openlibrary",
            "field": "subject_times",
            "status": "to_integrate",
            "interest": "historical context",
        },
        {
            "source": "openlibrary",
            "field": "cover_small / cover_medium / cover_large",
            "status": "to_integrate",
            "interest": "user interface",
        },
        {
            "source": "bnf",
            "field": "bnf_ark",
            "status": "to_integrate",
            "interest": "external identifier",
        },
        {
            "source": "bnf",
            "field": "page_count from format",
            "status": "to_study",
            "interest": "metadata",
        },
        {
            "source": "google_books",
            "field": "textSnippet",
            "status": "to_study",
            "interest": "description fallback",
        },
    ]
)

normalization_improvements

,source,field,status,interest
0,openlibrary,subjects,to_integrate,recommendation / rag
1,openlibrary,subject_people,to_integrate,characters / universe
2,openlibrary,subject_places,to_integrate,context
3,openlibrary,subject_times,to_integrate,historical context
4,openlibrary,cover_small / cover_medium / cover_large,to_integrate,user interface
5,bnf,bnf_ark,to_integrate,external identifier
6,bnf,page_count from format,to_study,metadata
7,google_books,textSnippet,to_study,description fallback


## Export de la synthèse

In [24]:

output_dir = PROJECT_ROOT / "data" / "processed" / "source_quality_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

coverage_by_source.to_csv(output_dir / "coverage_by_source.csv", index=False)
metadata_completeness.to_csv(output_dir / "metadata_completeness_by_source.csv")
df_coverage_by_isbn.to_csv(output_dir / "coverage_by_isbn_multisource.csv", index=False)
error_summary.to_csv(output_dir / "error_summary.csv", index=False)

output_dir

WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/processed/source_quality_analysis')

In [29]:
normalization_improvements.to_csv(
    output_dir / "normalization_improvements.csv",
    index=False,
)